Archived research source. Outputs and historical data have been removed. These cells reference the pre-v2 layout; use the documented nflsim APIs and commands for current analysis.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
year = 2024

### OFFENSE

In [ ]:
nfl_teams=["ARI", "ATL", "BAL", "BUF", "CAR", "CHI", "CIN", "CLE", "DAL", "DEN", 
           "DET", "GB", "HOU", "IND", "JAX", "KC", "LV", "LAC", "LA", "MIA", 
           "MIN", "NE", "NO", "NYG", "NYJ", "PHI", "PIT", "SF", "SEA", "TB", 
           "TEN", "WAS"]

ep_sets = []
down = 1
distance = 10

unfinished = []
for team in nfl_teams:
    try:
        ep_set_df = pd.read_csv('team_data/team-data'+str(year)+'/'+team+'/norm_eps.csv')
        ep_set = ep_set_df[ep_set_df['Down'] == down]
        ep_set = ep_set[((ep_set['Distance'] == distance) & (ep_set['Yardline'] >= distance)) | ((ep_set['Yardline'] < distance) 
                                                                                                    & (ep_set['Distance'] == ep_set['Yardline']))]
        ep_sets.append((team, ep_set))
    except:
        unfinished.append(team)
        pass

nfl = pd.read_csv('league_data/NFL'+str(year)+'/final_eps.csv')
ep_set_nfl = nfl[nfl['Down'] == down]
ep_set_nfl = ep_set_nfl[((ep_set_nfl['Distance'] == distance) & (ep_set_nfl['Yardline'] >= distance)) | ((ep_set_nfl['Yardline'] < distance) 
                                                                                                   & (ep_set_nfl['Distance'] == ep_set_nfl['Yardline']))]
#ep_sets.append(("NFL", ep_set_nfl))

print(unfinished)

In [ ]:
# Sort ep_sets by the sum of EP in ascending order
sorted_ep_sets = sorted(ep_sets, key=lambda ep_set: np.sum(ep_set[1]['EP']), reverse=True)

snamelist=[]

# Print names in order
for name, _ in sorted_ep_sets:
    snamelist.append(name)

print(snamelist)

In [ ]:
plt.figure(figsize=(10, 12))

for ep_set in ep_sets:
    name, set = ep_set
    plt.scatter(set['Yardline'], set['EP'], label = name+": "+str(int(np.sum(set['EP']))))

plt.scatter(ep_set_nfl['Yardline'], ep_set_nfl['EP'], label = "NFL: "+str(int(np.sum(ep_set_nfl['EP']))))
plt.legend()

In [ ]:

plt.figure(figsize=(8, 20))

# Plot scatter points and regression lines
for ep_set in ep_sets:
    name, set = ep_set
    
    # Compute linear regression line
    coeffs = np.polyfit(set['Yardline'], set['EP'], 1)
    x_vals = np.linspace(min(set['Yardline']), max(set['Yardline']), 100)
    y_vals = np.polyval(coeffs, x_vals)
    
    # Plot regression line
    plt.plot(x_vals, y_vals, linestyle="--", label=f"{name} Regression")

    # Label the line at its rightmost point
    plt.text(x_vals[-1], y_vals[-1], f"{name} Regression", verticalalignment='bottom', fontsize=10)

# Compute and plot NFL regression line
nfl_coeffs = np.polyfit(ep_set_nfl['Yardline'], ep_set_nfl['EP'], 1)
x_vals_nfl = np.linspace(min(ep_set_nfl['Yardline']), max(ep_set_nfl['Yardline']), 100)
y_vals_nfl = np.polyval(nfl_coeffs, x_vals_nfl)

# Plot NFL regression line
plt.plot(x_vals_nfl, y_vals_nfl, linestyle="--", label="NFL Regression")

# Label the NFL line at its rightmost point
plt.text(x_vals_nfl[-1], y_vals_nfl[-1], "NFL Regression", verticalalignment='bottom', fontsize=10)

plt.xlabel("Yardline")
plt.ylabel("EP")
plt.title("Expected Points vs. Yardline with Linear Regression")
plt.show()

In [ ]:
# Create a list of (name, total_EP) tuples
ep_sums = [(name, np.sum(df['EP'])) for name, df in ep_sets]

# Sort by total_EP
ep_sums_sorted = sorted(ep_sums, key=lambda x: x[1], reverse=True)  # reverse=True for descending

# Print results
for name, total in ep_sums_sorted:
    print(f"{name}: {total:.2f}")


In [ ]:
freq_df = pd.read_csv("naive_frequency.csv")

In [ ]:
# Rename for consistency
freq_df.rename(columns={'down': 'Down', 'distance': 'Distance', 'yardline': 'Yardline'}, inplace=True)

# Apply the same filtering logic to match df in ep_sets:
# - down == 1
# - (distance == 10 and yardline >= 10) OR (distance == yardline and yardline < 10)
freq_df_filtered = freq_df[
    (freq_df['Down'] == 1) &
    (
        ((freq_df['Distance'] == 10) & (freq_df['Yardline'] >= 10)) |
        ((freq_df['Distance'] == freq_df['Yardline']) & (freq_df['Yardline'] < 10))
    )
]

tot = np.sum(freq_df_filtered['frequency'])

# Now compute weighted sums
ep_sums = []
for name, df in ep_sets:
    # Merge EP and frequency data on Down, Distance, Yardline
    merged = pd.merge(df, freq_df_filtered, on=['Down', 'Distance', 'Yardline'], how='inner')

    # Compute weighted EP
    merged['weighted_ep'] = merged['EP'] * merged['frequency']
    total_weighted_ep = merged['weighted_ep'].sum()

    ep_sums.append((name, total_weighted_ep/tot))

# Sort and print
ep_sums_sorted = sorted(ep_sums, key=lambda x: x[1], reverse=True)
rank = 1
for name, total in ep_sums_sorted:
    print(f"{rank:2d}: {name:<3}: {total:.2f}")
    rank+=1

### DEFENSE

In [ ]:
def_ep_sets = []
down = 1
distance = 10

unfinished = []

for team in nfl_teams:
    try:
        def_ep_set_df = pd.read_csv('team_data/team-data'+str(year)+'/'+team+'/norm_def_eps.csv')
        def_ep_set = def_ep_set_df[def_ep_set_df['Down'] == down]
        def_ep_set = def_ep_set[((def_ep_set['Distance'] == distance) & (def_ep_set['Yardline'] >= distance)) | ((def_ep_set['Yardline'] < distance) 
                                                                                                    & (def_ep_set['Distance'] == def_ep_set['Yardline']))]
        def_ep_sets.append((team, def_ep_set))
    except:
        unfinished.append(team)
        pass

print(unfinished)

In [ ]:
# Sort ep_sets by the sum of EP in ascending order
sorted_ep_sets = sorted(def_ep_sets, key=lambda ep_set: np.sum(ep_set[1]['EP']), reverse=True)

snamelist=[]

# Print names in order
for name, _ in sorted_ep_sets:
    snamelist.append(name)

print(snamelist)

In [ ]:
plt.figure(figsize=(10, 12))

for ep_set in def_ep_sets:
    name, set = ep_set
    plt.scatter(set['Yardline'], set['EP'], label = name+": "+str(int(np.sum(set['EP']))))

plt.scatter(ep_set_nfl['Yardline'], ep_set_nfl['EP'], label = "NFL: "+str(int(np.sum(ep_set_nfl['EP']))))
plt.legend()

In [ ]:

plt.figure(figsize=(8, 20))

# Plot scatter points and regression lines
for ep_set in def_ep_sets:
    name, set = ep_set
    
    # Compute linear regression line
    coeffs = np.polyfit(set['Yardline'], set['EP'], 1)
    x_vals = np.linspace(min(set['Yardline']), max(set['Yardline']), 100)
    y_vals = np.polyval(coeffs, x_vals)
    
    # Plot regression line
    plt.plot(x_vals, y_vals, linestyle="--", label=f"{name} Regression")

    # Label the line at its rightmost point
    plt.text(x_vals[-1], y_vals[-1], f"{name} Regression", verticalalignment='bottom', fontsize=10)

# Compute and plot NFL regression line
nfl_coeffs = np.polyfit(ep_set_nfl['Yardline'], ep_set_nfl['EP'], 1)
x_vals_nfl = np.linspace(min(ep_set_nfl['Yardline']), max(ep_set_nfl['Yardline']), 100)
y_vals_nfl = np.polyval(nfl_coeffs, x_vals_nfl)

# Plot NFL regression line
plt.plot(x_vals_nfl, y_vals_nfl, linestyle="--", label="NFL Regression")

# Label the NFL line at its rightmost point
plt.text(x_vals_nfl[-1], y_vals_nfl[-1], "NFL Regression", verticalalignment='bottom', fontsize=10)

plt.xlabel("Yardline")
plt.ylabel("EP")
plt.title("Expected Points vs. Yardline with Linear Regression")
plt.show()

In [ ]:
# Create a list of (name, total_EP) tuples
def_ep_sums = [(name, np.sum(df['EP'])) for name, df in def_ep_sets]

# Sort by total_EP
def_ep_sums_sorted = sorted(def_ep_sums, key=lambda x: x[1], reverse=False)  # reverse=False for ascending

# Print results
for name, total in def_ep_sums_sorted:
    print(f"{name}: {total:.2f}")

In [ ]:
freq_df = pd.read_csv("naive_frequency.csv")

# Rename for consistency
freq_df.rename(columns={'down': 'Down', 'distance': 'Distance', 'yardline': 'Yardline'}, inplace=True)

# Apply the same filtering logic to match df in ep_sets:
# - down == 1
# - (distance == 10 and yardline >= 10) OR (distance == yardline and yardline < 10)
freq_df_filtered = freq_df[
    (freq_df['Down'] == 1) &
    (
        ((freq_df['Distance'] == 10) & (freq_df['Yardline'] >= 10)) |
        ((freq_df['Distance'] == freq_df['Yardline']) & (freq_df['Yardline'] < 10))
    )
]

tot = np.sum(freq_df_filtered['frequency'])

# Now compute weighted sums
def_ep_sums = []
for name, df in def_ep_sets:
    # Merge EP and frequency data on Down, Distance, Yardline
    merged = pd.merge(df, freq_df_filtered, on=['Down', 'Distance', 'Yardline'], how='inner')

    # Compute weighted EP
    merged['weighted_ep'] = merged['EP'] * merged['frequency']
    total_weighted_ep = merged['weighted_ep'].sum()

    def_ep_sums.append((name, total_weighted_ep/tot))

# Sort and print
def_ep_sums_sorted = sorted(def_ep_sums, key=lambda x: x[1], reverse=False)
rank = 1
for name, total in def_ep_sums_sorted:
    print(f"{rank:2d}: {name:<3}: {total:.2f}")
    rank+=1

### Cumulative Ranking

In [ ]:
# Convert both lists to dictionaries
off_dict = dict(ep_sums_sorted)
def_dict = dict(def_ep_sums_sorted)

# Compute the difference per team (offense - defense)
diff_sums = []
for team in off_dict:
    diff = off_dict[team] - def_dict.get(team, 0)
    diff_sums.append((team, diff))

# Sort and print the differences in descending order
diff_sums_sorted = sorted(diff_sums, key=lambda x: x[1], reverse=True)

for rank, (team, diff) in enumerate(diff_sums_sorted, start=1):
    print(f"{rank:2d}: {team:<3}: {diff:.2f}")

In [ ]:
# Convert each list to a DataFrame
off_df = pd.DataFrame(ep_sums_sorted, columns=["Team", "Offensive AEP"])
def_df = pd.DataFrame(def_ep_sums_sorted, columns=["Team", "Defensive AEP"])

# Merge on Team
merged_df = pd.merge(off_df, def_df, on="Team")

# Sort alphabetically by team name
merged_df = merged_df.sort_values(by="Team").reset_index(drop=True)

print(merged_df)

merged_df.to_csv('AEPs/nfl'+str(year)+'.csv')


In [ ]:
# Output TypeScript-style arrays for use in your frontend

offense_vals = merged_df["Offensive AEP"].tolist()
defense_vals = merged_df["Defensive AEP"].tolist()

# Format as exportable JS arrays
def format_array(name, arr):
    formatted = ", ".join(f"{x:.2f}" for x in arr)
    return f"export const {name} = [\n    {formatted}\n]"

print(format_array("offenseValues"+str(year), offense_vals))
print(format_array("defenseValues"+str(year), defense_vals))
